In [1]:
import pandas as pd

df = pd.read_csv(
    r"C:\Users\HP\Internship\DataCoSupplyChainDataset.csv",
    encoding="cp1252"
)

print(df.shape)
print(df.columns.tolist())

(180519, 53)
['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Delivery Status', 'Late_delivery_risk', 'Category Id', 'Category Name', 'Customer City', 'Customer Country', 'Customer Email', 'Customer Fname', 'Customer Id', 'Customer Lname', 'Customer Password', 'Customer Segment', 'Customer State', 'Customer Street', 'Customer Zipcode', 'Department Id', 'Department Name', 'Latitude', 'Longitude', 'Market', 'Order City', 'Order Country', 'Order Customer Id', 'order date (DateOrders)', 'Order Id', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Order Region', 'Order State', 'Order Status', 'Order Zipcode', 'Product Card Id', 'Product Category Id', 'Product Description', 'Product Image', 'Product Name', 'Product Price', 'Product Status', 'shipping d

In [2]:
cols = [
    "Days for shipping (real)",
    "Days for shipment (scheduled)",
    "order date (DateOrders)",
    "shipping date (DateOrders)"
]

for col in cols:
    print(col, ":", col in df.columns)

Days for shipping (real) : True
Days for shipment (scheduled) : True
order date (DateOrders) : True
shipping date (DateOrders) : True


In [4]:
# 1. Convert date columns into datetime format
df["order date (DateOrders)"] = pd.to_datetime(
    df["order date (DateOrders)"], errors="coerce"
)

df["shipping date (DateOrders)"] = pd.to_datetime(
    df["shipping date (DateOrders)"], errors="coerce"
)
# 2. Delivery Duration
df["Delivery Duration"] = df["Days for shipping (real)"]

# 3. Delay compared with scheduled shipping duration
df["Delay Days"] = (
    df["Days for shipping (real)"]
    - df["Days for shipment (scheduled)"]
)

# 4. Order Processing Time
df["Order Processing Time"] = (
    df["shipping date (DateOrders)"]
    - df["order date (DateOrders)"]
).dt.total_seconds() / 86400

# 5. Calculate summary KPIs
summary = pd.DataFrame({
    "KPI": [
        "Average Delivery Duration",
        "Average Delay",
        "Average Order Processing Time"
    ],
    "Value (Days)": [
        df["Delivery Duration"].mean(),
        df["Delay Days"].mean(),
        df["Order Processing Time"].mean()
    ]
})

print(summary.round(2))

                             KPI  Value (Days)
0      Average Delivery Duration          3.50
1                  Average Delay          0.57
2  Average Order Processing Time          3.50


In [5]:
output_file = r"C:\Users\HP\Internship\07_Time_Based_KPI_Analysis.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    summary.round(2).to_excel(
        writer, sheet_name="KPI Summary", index=False
    )

    df[[
        "Order Id",
        "Days for shipping (real)",
        "Days for shipment (scheduled)",
        "Delivery Duration",
        "Delay Days",
        "order date (DateOrders)",
        "shipping date (DateOrders)",
        "Order Processing Time"
    ]].to_excel(
        writer, sheet_name="Detailed Analysis", index=False
    )

print("Excel file created:", output_file)

Excel file created: C:\Users\HP\Internship\07_Time_Based_KPI_Analysis.xlsx


In [6]:
print("Order Date:")
print(df["order date (DateOrders)"].head())

print("\nShipping Date:")
print(df["shipping date (DateOrders)"].head())

print("\nAverage Order-to-Shipping Time:")
print(df["Order Processing Time"].mean())

Order Date:
0   2018-01-31 22:56:00
1   2018-01-13 12:27:00
2   2018-01-13 12:06:00
3   2018-01-13 11:45:00
4   2018-01-13 11:24:00
Name: order date (DateOrders), dtype: datetime64[ns]

Shipping Date:
0   2018-02-03 22:56:00
1   2018-01-18 12:27:00
2   2018-01-17 12:06:00
3   2018-01-16 11:45:00
4   2018-01-15 11:24:00
Name: shipping date (DateOrders), dtype: datetime64[ns]

Average Order-to-Shipping Time:
3.498825608384713
